# Neural Networks

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

## Outline

- What networks do, and a little history
- One hidden layer
- How a network learns: backpropagation by hand, the dials
- Networks on real data: the homes, then a finance application, Gu,
  Kelly & Xiu (2020)
- Networks with memory: the LSTM
- What networks can do that forests and boosted trees cannot

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `ames_housing.csv`, `size_value_monthly.csv` and
`gw_monthly.csv` from Brightspace, drag them into the Files pane (folder
icon on the left), then run the two **Setup** cells below before any
other cell.

## Setup: What It Builds

- **`homes`** and **`log_price`**: 1,460 home sales in Ames, Iowa,
  2006–2010: 79 features each, and the price in logs.
- **`french`**: Kenneth French’s monthly portfolios and factors,
  1963–2025, for one beta.
- **`monthly`**: the Goyal–Welch file: the stock market’s monthly return
  over T-bills, from 1926.

<!-- -->

- **Your Turn** code is text in this notebook: click **+ Code**, paste
  it, fill the blank, then Shift + Enter.

Run this first, every session.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import warnings
import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import load_digits          # scikit-learn's handwritten digits: no file
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor   # scikit-learn's plain networks
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
plt.rcParams["font.size"] = 18     # big text for the projector

# Ames, Iowa home sales, 2006-2010 (Dean De Cock, Journal of Statistics Education, 2011).
homes = pd.read_csv(resolve_data_path("ames_housing.csv"))
log_price = np.log(homes.pop("SalePrice"))     # log dollars: an error is about a percent

# Kenneth French's 25 portfolios formed on size and book-to-market (value-weighted), with
# the market factor and the T-bill, per month, July 1963 to December 2025.
# Data: Copyright Eugene F. Fama and Kenneth R. French (French's Data Library).
# Returns are SIMPLE, in decimals per month (0.01 = 1%). mkt_rf: the market's EXCESS
# return (over the T-bill, rf). The portfolios are RAW returns: subtract rf for excess.
french = pd.read_csv(resolve_data_path("size_value_monthly.csv"))
french.index = pd.to_datetime(french["yyyymm"].astype(str), format="%Y%m")

# The Goyal-Welch monthly file of week 2. equity = stocks minus T-bills, simple, in decimals.
monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
monthly.index = pd.to_datetime(monthly["yyyymm"].astype(str), format="%Y%m")
print(homes.shape, french.shape, monthly.shape)

Then run this cell too, every session: helpers that later parts use, so
that each part also runs on its own after these two cells.

In [ ]:
def make_curve():
    """The curve of "Hockey Sticks Add Up to Curves": a curved truth, sin(3x), and 300 noisy
    observations of it, x between -1 and 1. The same draw on every run (start 5)."""
    generator = np.random.default_rng(5)
    x = np.sort(generator.uniform(-1, 1, 300))
    truth = np.sin(3 * x)
    y = truth + generator.normal(0, 0.3, 300)
    return x, truth, y


def fit_digit_network():
    """The digits network of "A Picture Is a Grid of Numbers", fitted again: three quarters of
    the digits to fit. Returns the quarter it never sees (the digits and their labels) and the
    fitted network."""
    digits = load_digits()
    digits_fit, digits_new, labels_fit, labels_new = train_test_split(
        digits.data, digits.target, test_size=0.25, random_state=0)
    return digits_new, labels_new, MLPClassifier(random_state=0).fit(digits_fit, labels_fit)


def walk_downhill(step, epochs=60, neurons=20, start=1):
    """Plain gradient descent (new weight = old - step x blame) for one hidden layer on the
    curve's 300 points. Every step uses every row, so a step is an epoch. Returns the miss on
    the 300 points before the first step and after each epoch; stops once it passes a million."""
    x, _, y = make_curve()                         # the points; the true curve is not needed
    generator = np.random.default_rng(start)
    edge = np.sqrt(6 / (1 + neurons))              # small random starting weights
    w_in, b_in = generator.uniform(-edge, edge, neurons), generator.uniform(-edge, edge, neurons)
    w_out, b_out = generator.uniform(-edge, edge, neurons), generator.uniform(-edge, edge)
    misses = []
    for epoch in range(epochs + 1):
        sums = np.outer(x, w_in) + b_in                    # add
        outs = np.maximum(0, sums)                         # bend
        gap = outs @ w_out + b_out - y
        misses.append(np.mean(gap ** 2))
        if epoch == epochs or misses[-1] > 1e6:            # done, or run away
            break
        blame = 2 * gap / len(x)                           # the blame at each row's forecast
        blame_sums = np.outer(blame, w_out) * (sums > 0)   # passed back: x 1 if on, x 0 if off
        w_out, b_out = w_out - step * outs.T @ blame, b_out - step * blame.sum()
        w_in, b_in = w_in - step * x @ blame_sums, b_in - step * blame_sums.sum(axis=0)
    return np.array(misses)


# The dials' few rows: 20 of the curve's 300 points (every 15th) to fit, the other 280 held out.
curve_x, _, curve_y = make_curve()
every_15th = np.arange(300) % 15 == 0
few_X, few_y = curve_x[every_15th].reshape(-1, 1), curve_y[every_15th]
held_X, held_y = curve_x[~every_15th].reshape(-1, 1), curve_y[~every_15th]


def few_network(layers, penalty=0.0, start=0):
    """A network fitted to the 20 rows: a step of 0.01, 600 epochs with no stopping rule, and
    the penalty λ (scikit-learn's alpha), none unless told."""
    network = MLPRegressor(hidden_layer_sizes=layers, alpha=penalty, learning_rate_init=0.01,
                           max_iter=600, tol=0, n_iter_no_change=601, random_state=start)
    with warnings.catch_warnings():               # 600 epochs on purpose: no "not converged"
        warnings.simplefilter("ignore", ConvergenceWarning)
        return network.fit(few_X, few_y)


def fitting_miss(network):
    """The average squared gap between a network's forecasts and the 20 points it fitted."""
    return np.mean((network.predict(few_X) - few_y) ** 2)


def held_out_miss(network):
    """The average squared gap between a network's forecasts and the 280 held-out points."""
    return np.mean((network.predict(held_X) - held_y) ** 2)

## What a Network Does

| Application | Inputs → output |
|----------------------|--------------------------------------------------|
| ZIP codes, checks | a handwritten digit’s pixels → which digit |
| Medical images | a photo of the eye → signs of diabetic eye disease |
| Speech to text | sound → the words spoken |
| Translation | a Chinese sentence → the English one |
| ChatGPT | the words so far → the next word |
| AlphaGo (2016) | a Go board → a good next move |
| AlphaFold | a protein’s chain of amino acids → its 3-D shape |
| Card fraud alerts | a card payment → fraud or not |
| Gu, Kelly & Xiu (2020) | features (firm characteristics) → next month’s return |

- **Every row: one flexible function** of the inputs, fitted to make the
  **miss** small.
- **The last row is finance.** We come back to it in detail.

## A Picture Is a Grid of Numbers

- **scikit-learn’s built-in digits** (`load_digits`): small handwritten
  digits, each a grid of squares of ink, from white to black.
- **So a picture is a list of numbers,** the inputs. The output: which
  digit.
- **A plain network** (`MLPClassifier`: every square feeds every neuron)
  reads almost all of the digits it never saw.
- **Yann LeCun’s networks** (Bell Labs) read ZIP codes, then bank checks
  from 1993: over 10% of US checks by the late 1990s.

**Try it.** The whole network is one line, `MLPClassifier().fit(...)`:
scikit-learn’s plain network, one hidden layer of 100 neurons. It learns
from three quarters of the digits and is scored on the quarter it never
saw.

In [ ]:
# scikit-learn's built-in handwritten digits: no file to upload. Each digit is a grid of
# 8 x 8 squares, each square's ink from 0 (white) to 16 (black). MLPClassifier: scikit-learn's
# plain network for classes (imported in Setup).
digits = load_digits()
print(digits.images[0].astype(int))                # the first digit, a 0, as numbers

# digits.data: each grid as one row of 64 numbers; digits.target: which digit it is.
digits_fit, digits_new, labels_fit, labels_new = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=0)
digit_network = MLPClassifier(random_state=0).fit(digits_fit, labels_fit)
print(f"Share of new digits read right: {digit_network.score(digits_new, labels_new):.0%}")

## Convolution: Slide a Small Detector

- **A detector:** a small grid of **weights** (numbers a network learns)
  that score one pattern, such as an edge where white turns to ink.
- **Slide it over every window,** the same weights: the result, a map,
  is high wherever the pattern is.
- **A convolutional network** learns its detectors, stacked in layers:
  edges, then strokes, then digits. LeCun’s LeNet (1989, 1998); the 2018
  Turing Award, with Hinton and Bengio.

**Try it.** Slide the detector over the 0 by hand: two loops, one window
at a time. Then try another detector.

In [ ]:
# A convolution by hand. The detector: -1 on the left column, +1 on the right, so it is high
# where white turns to ink going right (the left edge of a stroke).
detector = np.array([[-1, 0, 1],
                     [-1, 0, 1],
                     [-1, 0, 1]])
zero = digits.images[0].astype(int)                # the first digit, a 0: 8 x 8 numbers
edge_map = np.zeros((6, 6), dtype=int)             # one number per 3 x 3 window
for row in range(6):                               # slide the window down ...
    for col in range(6):                           # ... and across
        window = zero[row:row + 3, col:col + 3]
        edge_map[row, col] = np.sum(window * detector)   # multiply square by square, add
print(edge_map)
# Try [[-1, -1, -1], [0, 0, 0], [1, 1, 1]]: high where white turns to ink going down.

## A Short History

- **1958:** the perceptron (Rosenblatt). **1969:** one layer’s limits
  (Minsky & Papert). **1986:** backpropagation (Rumelhart, Hinton &
  Williams). **1989:** universal approximation (Cybenko; Hornik,
  Stinchcombe & White). **1990s:** networks read handwritten checks
  (LeCun). **1997:** the LSTM, networks with memory (Hochreiter &
  Schmidhuber). **2012:** AlexNet wins at images, on graphics chips.
  **2017:** the transformer (Vaswani et al.). **2022:** ChatGPT.
- **The ideas are old.** What is new: the data (millions of labeled
  images, then the text of the web), graphics chips, and better tricks
  for training.
- **Two quiet spells for networks** in between, when other methods won.
- **Finance has less data:** about a hundred years of months for one
  market, and a tiny signal in each stock.

## Universal Approximation

- **1989, two proofs:** Cybenko; Hornik, Stinchcombe & White. Halbert
  White was an econometrician.
- **In plain words:** a network with one **hidden layer** (one layer of
  simple units, **neurons**, between inputs and output) can come as
  close as you like to any continuous curve over a bounded range, given
  enough neurons.
- **Leshno et al. (1993):** the same for almost any shape a neuron uses,
  including the simple one modern networks are built from.
- **It does not promise:**
  - how many neurons, or how much data;
  - that training finds the right weights;
  - anything outside the range of the data.
- **We come back to it** once neurons are built.

# One Hidden Layer

**Starting here?** Run the Setup cells at the top first, then this cell.
It rebuilds what this part uses from earlier parts: the digits network
of “A Picture Is a Grid of Numbers”.

In [ ]:
digits_new, labels_new, digit_network = fit_digit_network()

## One Neuron: Add, Then Bend

In [ ]:
# One neuron: a weighted sum of the features (add), then max(0, z) (bend). The weights
# are made up, to show the arithmetic; a network learns them from data.
weights = {"size": -0.4, "value": 0.6, "momentum": 0.5}
intercept = 0.1
stocks = {"stock A": {"size": -0.5, "value": 0.8, "momentum": 0.2},
          "stock B": {"size": 0.5, "value": -0.8, "momentum": -0.2}}
sums = {name: intercept + sum(weights[key] * value for key, value in row.items())
        for name, row in stocks.items()}              # the "add" step
outputs = {name: max(0, z) for name, z in sums.items()}   # the "bend" step

z_grid = np.linspace(-1.2, 1.2, 100)
plt.figure(figsize=(10, 2.3))
plt.plot(z_grid, np.maximum(0, z_grid), color="tab:orange", linewidth=3)
for name, z in sums.items():
    plt.scatter(z, outputs[name], s=120, color="black", zorder=3)
    plt.text(z, outputs[name] + 0.2, name, ha="center")
plt.xlabel("z: the weighted sum")
plt.ylabel("neuron output")
for side in ["top", "right"]:
    plt.gca().spines[side].set_visible(False)
plt.show()

- **Add:**
  $z = 0.1 - 0.4\,\text{size} + 0.6\,\text{value} + 0.5\,\text{momentum}$.
- **The weights** −0.4, 0.6 and 0.5 work like a regression’s
  coefficients; 0.1 is the **intercept** (“bias” in the readings).
  Training chooses them.
- **Bend:** output $= \max(0, z)$, the **ReLU** (the bend is called the
  **activation**): a negative sum is cut to zero, and the neuron is off.

## A Network Is Stacked Regressions

- **Each hidden neuron:** a weighted sum of the inputs, like a
  regression’s forecast, then the bend: a hockey stick.
- **The forecast:** a weighted sum of the neurons.
- **Each link is one weight** (every input to every neuron, every neuron
  to the forecast); all are fitted together, not one regression at a
  time.
- **Without the bend,** the stack collapses to one regression: the bends
  do the work.
- **Deep:** several hidden layers, stacked.

**Try it.** The digits network from above, opened up: how many weights
each layer of links holds, and how many intercepts.

In [ ]:
# A fitted network keeps its weights in coefs_ (one table per layer of links: every input to
# every neuron) and its intercepts in intercepts_ (one per neuron).
for layer_weights in digit_network.coefs_:
    links_from, links_to = layer_weights.shape
    print(f"from {links_from} to {links_to}: {layer_weights.size:,} weights")
intercept_count = sum(layer.size for layer in digit_network.intercepts_)
print(f"intercepts: {intercept_count}")
weight_count = sum(layer.size for layer in digit_network.coefs_)
print(f"in all: {weight_count + intercept_count:,} numbers, every one chosen in training")

## Hockey Sticks Add Up to Curves

In [ ]:
# The curve of the trees video ("One Round of Boosting"): one input x, a curved truth, and
# 300 noisy observations of it.
curve_generator = np.random.default_rng(5)
x_curve = np.sort(curve_generator.uniform(-1, 1, 300))
truth_curve = np.sin(3 * x_curve)
y_curve = truth_curve + curve_generator.normal(0, 0.3, 300)
X_curve = x_curve.reshape(-1, 1)                  # scikit-learn wants a table: one column


def miss(fitted):
    """How far a fitted curve is from the truth: the average squared gap."""
    return np.mean((fitted - truth_curve) ** 2)


line_fit = LinearRegression().fit(X_curve, y_curve).predict(X_curve)
tree_fit = DecisionTreeRegressor(max_depth=3).fit(X_curve, y_curve).predict(X_curve)
# Four hockey sticks, max(0, x - bend), with the bends FIXED by us at -0.75, -0.25, 0.25
# and 0.75; OLS chooses how much of each to add up.
bends = [-0.75, -0.25, 0.25, 0.75]
sticks = np.column_stack([np.maximum(0, x_curve - bend) for bend in bends])
sticks_fit = LinearRegression().fit(sticks, y_curve).predict(sticks)

figure, axes = plt.subplots(1, 3, figsize=(11, 2.6), sharey=True)
panels = [("line", line_fit, "tab:blue"), ("tree", tree_fit, "tab:green"),
          ("sticks", sticks_fit, "tab:orange")]
for axis, (name, fitted, color) in zip(axes, panels):
    axis.scatter(x_curve, y_curve, color="lightgray", s=10)
    axis.plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
    axis.plot(x_curve, fitted, color=color, linewidth=3)
    axis.set_title(f"{name}: miss {miss(fitted):.3f}")
    axis.set_xticks([])
    axis.set_yticks([])
for bend in bends:
    axes[2].axvline(bend, color="gray", linestyle=":", linewidth=1.5)
plt.subplots_adjust(wspace=0.05)
plt.show()

- **Miss:** the average squared error against the true curve (dashed),
  known only because we simulated it. Lines, then steps, then bends.
- Four hockey sticks, max(0, x − bend), bends fixed by us (dotted),
  weights from OLS: about as close as the tree. A network **chooses the
  bends itself**.

## More Sticks, Closer Fit

- **“Universal Approximation”, in one sentence:** more sticks, closer to
  the true curve, inside the range of the data.

**Try it.** More sticks, the same 300 points: the bends sit at the
middles of equal pieces of the range, and OLS adds the sticks up. Then
try other counts.

In [ ]:
# 2, 5 and 20 hockey sticks. The miss is against the true curve, as in the last cell.
for stick_count in [2, 5, 20]:                     # try 3, 50 or 200
    stick_bends = [-1 + (2 * piece + 1) / stick_count for piece in range(stick_count)]
    stick_table = np.column_stack([np.maximum(0, x_curve - bend) for bend in stick_bends])
    stick_fit = LinearRegression().fit(stick_table, y_curve).predict(stick_table)
    print(f"{stick_count:>3} sticks: miss {miss(stick_fit):.3f}")

## Check: No Bend

- A network with one hidden layer of 100 neurons, but no bend: each
  neuron passes its weighted sum straight on.
- **Fingers up:** it is the same as:
  1.  A linear regression
  2.  A curve with 100 bends
  3.  A tree with 100 leaves
  4.  The historical mean

**The check, by computer.** Vote first, then run it: the network of the
check, 100 neurons with no bend (`activation="identity"`), on the 300
points of the curve, next to OLS’s line.

In [ ]:
# 100 neurons, no bend: each neuron passes its weighted sum straight on.
no_bend_network = MLPRegressor(hidden_layer_sizes=(100,), activation="identity",
                               learning_rate_init=0.01, max_iter=5000, random_state=0)
no_bend_fit = no_bend_network.fit(X_curve, y_curve).predict(X_curve)
plt.figure(figsize=(10, 3))
plt.scatter(x_curve, y_curve, color="lightgray", s=8)
plt.plot(x_curve, no_bend_fit, color="tab:orange", linewidth=5, label="100 neurons, no bend")
plt.plot(x_curve, line_fit, color="black", linestyle="--", linewidth=2, label="OLS")
plt.legend(frameon=False)
plt.show()

# How a Network Learns

**Starting here?** Run the Setup cells at the top first, then this cell.
It rebuilds what this part uses from earlier parts: the curve of “Hockey
Sticks Add Up to Curves”, its 300 points and the true curve.

In [ ]:
x_curve, truth_curve, y_curve = make_curve()
X_curve = x_curve.reshape(-1, 1)                  # scikit-learn wants a table: one column

## Training: The Same Problem as OLS

|             | OLS                    | A network                     |
|:------------|:-----------------------|:------------------------------|
| Chooses     | the coefficients       | the weights and intercepts    |
| To minimize | the mean squared error | the same, or another **loss** |
| Solved by   | one formula            | an algorithm, step by step    |

- **Still an optimization problem:** choose the numbers that make the
  loss smallest. For yes or no, as for the logit, the loss punishes a
  low probability on what happened.
- **Too hard for a formula:** thousands of weights, bends between them,
  and a loss with many valleys.
- **The algorithm, repeated:**
  1.  Forecast with the current weights.
  2.  Compute the loss.
  3.  **Backpropagation:** each weight’s slope.
  4.  Move every weight a step downhill.

## Predict: A Step Too Big

- **One weight, to see the walk:** a portfolio’s **beta**, how much it
  moves when the market moves.
- **The walk:** start at beta = 0. At each step, move beta against the
  **slope** of the miss: how much the miss rises if beta rises a little.
  The **step size** sets how far.
- **Predict first:** what does a step that is much too big do?
  1.  Gets to the bottom faster
  2.  Overshoots, and runs away
  3.  Stops halfway
  4.  Never moves

## Walk Downhill

In [ ]:
# One weight to learn: a portfolio's beta. Small value stocks (ME1_BM5: the smallest fifth of
# firms, the highest fifth of book-to-market) and the market: monthly EXCESS returns in
# percent, July 1963 to December 2025.
portfolio = 100 * (french["ME1_BM5"] - french["rf"])
market = 100 * french["mkt_rf"]


def beta_miss(beta):
    """The miss for one beta: the average squared gap between the portfolio and beta x market."""
    return np.mean((portfolio - beta * market) ** 2)


def beta_slope(beta):
    """The slope of the miss at beta: positive means the miss rises as beta grows."""
    return -2 * np.mean(market * (portfolio - beta * market))


walks = {}
for step in [0.004, 0.02, 0.055]:
    beta = 0.0                                  # start at zero
    walks[step] = [beta]
    for move in range(8):                       # eight steps
        beta = beta - step * beta_slope(beta)   # the rule: a step against the slope
        walks[step].append(beta)
best_beta = np.mean(market * portfolio) / np.mean(market ** 2)   # the bottom, by formula

figure, axes = plt.subplots(1, 3, figsize=(11, 2.7))
for axis, (step, verdict) in zip(axes, [(0.004, "too small"), (0.02, "about right"),
                                        (0.055, "too big")]):
    path = walks[step]
    grid = np.linspace(min(path + [best_beta]) - 0.3, max(path + [best_beta]) + 0.3, 200)
    axis.plot(grid, [beta_miss(beta) for beta in grid], color="gray", linewidth=2)
    axis.plot(path, [beta_miss(beta) for beta in path], color="tab:orange", marker="o",
              markersize=7, linewidth=1.5)
    axis.axvline(best_beta, color="black", linestyle=":", linewidth=1.5)
    axis.set_title(f"{step}: {verdict}", fontsize=17)
    axis.set_xlabel("beta, zoomed out" if step == 0.055 else "beta")
    axis.set_yticks([])
axes[0].set_ylabel("miss")
plt.subplots_adjust(wspace=0.08)
plt.show()

- **The rule:** new beta = beta − step × slope of the miss. Start at 0,
  take 8 steps. The dotted line is the bottom.
- **Too small:** still short after 8 steps. **About right:** at the
  bottom within a few steps.
- **Too big:** each step overshoots by more than the last, and beta runs
  away.

## Backpropagation by Hand: Forward

- **One stock,** two features (value and momentum scores), and a network
  of two neurons, A and B, with made-up weights and no intercepts.
- **Add, then bend,** for A and for B; then add up the forecast and
  compare it with what happened: the miss.

## Predict: Which Weights Move?

- The network above (value 1, momentum 2; A on, B off): the forecast is
  below what happened.
- Neuron A is on; neuron B is off.
- One step downhill moves each weight against its own slope of the miss.
- **Predict first:** after one step, which weights have moved?
  1.  All six
  2.  The three on A’s path
  3.  Only the two into the forecast
  4.  None: the forecast is close enough

## Passing the Blame Back

- **Blame:** how much the miss moves when one number rises a little.
- **Backpropagation does it right to left, once:** a weight’s blame =
  the blame where its arrow lands × what traveled along it.
- **A bend:** × 1 if its neuron is on, × 0 if off. An off neuron stops
  the blame, so its weights get none.

## One Step Downhill, Every Weight

- **The rule of “Walk Downhill”,** for every weight at once: new = old −
  step × blame.
- **Big blame, big change; no blame, no change:** an off neuron’s
  weights stay put.
- **Forward again:** the forecast moves toward what happened, and the
  miss shrinks. A step too big overshoots.

**Try it.** The network of the last three slides in numpy: add, bend,
pass the blame back, and take one step for every weight, three times.
Then try a step of 0.2.

In [ ]:
# The two-neuron network of the last three slides. One stock: value score 1, momentum score 2;
# it returned 3%. into_neurons: one row per input, its arrows to A and to B. into_forecast: the
# arrows from A and from B to the forecast. No intercepts.
stock_scores = np.array([1.0, 2.0])
actual_return = 3.0
step_size = 0.05                                    # try 0.2
into_neurons = np.array([[0.2, 0.6],                # value -> A, value -> B
                         [0.4, -0.5]])              # momentum -> A, momentum -> B
into_forecast = np.array([1.0, 0.5])                # A -> forecast, B -> forecast
hand_misses = []
for step_number in range(4):
    neuron_sums = stock_scores @ into_neurons       # add ...
    neuron_outs = np.maximum(0, neuron_sums)        # ... then bend
    forecast = neuron_outs @ into_forecast
    hand_misses.append((actual_return - forecast) ** 2)
    print(f"steps so far: {step_number}; forecast {forecast:.2f}%, miss {hand_misses[-1]:.3f}")
    blame_forecast = 2 * (forecast - actual_return)                     # the blame at the forecast
    blame_neurons = blame_forecast * into_forecast * (neuron_sums > 0)  # passed back; x 0 if off
    # One step for every weight: new = old - step x blame (the blame where the arrow lands x
    # what traveled along it).
    into_forecast = into_forecast - step_size * blame_forecast * neuron_outs
    into_neurons = into_neurons - step_size * np.outer(stock_scores, blame_neurons)

## Batches and Epochs

- **A real network:** the same sweep, right to left, gives the blame of
  every weight at once, thousands or millions of them. That is
  **backpropagation**; software does the bookkeeping.
- **Rows:** each row has its own blames. A **batch** is a few rows
  (scikit-learn: 200): average their blames, take one step.
- **An epoch:** one pass through all the fitting rows.
- **Why batches:** one row at a time is jumpy; all rows at once is slow.
  The rows are reshuffled every epoch: **stochastic gradient descent**
  (all the slopes together: the **gradient**).

**Try it.** A network with one hidden layer of 100 neurons, trained one
epoch at a time on the curve’s 300 rows. `partial_fit` walks once
through the rows, in batches of 200: two steps an epoch. The miss on the
fitting rows falls over the epochs.

In [ ]:
# One epoch at a time: scikit-learn's walker (Adam), a step of 0.01, start 0.
epoch_network = MLPRegressor(hidden_layer_sizes=(100,), learning_rate_init=0.01, random_state=0)
for epoch in range(1, 51):
    epoch_network.partial_fit(X_curve, y_curve)                # one more epoch: two batches
    if epoch <= 5 or epoch % 10 == 0:
        epoch_miss = np.mean((epoch_network.predict(X_curve) - y_curve) ** 2)
        print(f"after epoch {epoch:>2}: miss {epoch_miss:.3f} on the fitting rows")

## Predict: Four Neurons or a Hundred?

- The curve of the hockey sticks: 300 points, one input.
- Two networks with one hidden layer: **4** neurons, and **100**.
- Each is trained three times, from three random starting weights.
- **Predict first:** which finds the curve from every start?
  1.  Four neurons
  2.  A hundred neurons
  3.  Both
  4.  Neither

## A Network Finds the Bends Itself

In [ ]:
# Two networks with one hidden layer, each trained from three random starting weights
# (random_state). learning_rate_init sets the size of the steps ("How a Network Learns").
network_fits = {}
for neurons in [4, 100]:
    for start in [0, 1, 2]:
        network = MLPRegressor(hidden_layer_sizes=(neurons,), learning_rate_init=0.01,
                               max_iter=5000, random_state=start)
        network.fit(X_curve, y_curve)
        network_fits[neurons, start] = network.predict(X_curve)

figure, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for axis, neurons in zip(axes, [4, 100]):
    axis.scatter(x_curve, y_curve, color="lightgray", s=10)
    axis.plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
    for start, color in zip([0, 1, 2], ["tab:orange", "tab:blue", "tab:green"]):
        axis.plot(x_curve, network_fits[neurons, start], color=color, linewidth=2.5)
    axis.set_title(f"{neurons} neurons: three starts")
    axis.set_xticks([])
    axis.set_yticks([])
plt.subplots_adjust(wspace=0.05)
plt.show()

- **Four neurons:** the start decides: one start finds the curve, the
  others get stuck.
- **A hundred:** every start finds it, closer than the tree of depth 3.
- **No wiggles:** 300 points on one input are plenty. With few rows,
  networks do memorize: the dials, next.

## The Dials

| Dial | Too little | Too much |
|---------------------------|-----------------------|-----------------------|
| The step (learning rate) | creeps | overshoots, runs away |
| Neurons and layers | misses bends; starts get stuck | slow; can memorize |
| How long to train (epochs) | stops short | fits the noise |
| The penalty λ, as in ridge (`alpha` in scikit-learn) | weights chase noise | flattens the fit |
| Random starts, averaged | one start may be stuck | costs only time |
| Batch size | each step jumpy | slow steps |

- One section per dial next; batch size was “Batches and Epochs”.
- Choose every dial on **validation data**, never on the test rows.

## Dial: The Step

- **A training curve:** the miss on the fitting rows after each epoch.
- **Too small:** still creeping. **About right:** falls fast, then
  flattens.
- **Too big:** down once, then up, and it runs away: each step
  overshoots by more.
- In scikit-learn: `learning_rate_init`.

**Try it.** The training curves of this slide, as numbers.
`walk_downhill`, in the second Setup cell, is the plain walk downhill:
20 neurons on the curve’s 300 points, every row in every step. Then try
a step of 0.3: the miss jumps up and down, but does not run away.

In [ ]:
# Three step sizes, the same start: the miss before the first step and at the end.
for walk_step in [0.01, 0.1, 1.5]:
    step_misses = walk_downhill(walk_step)          # the miss at the start, then after each epoch
    print(f"step {walk_step}: miss {step_misses[0]:.2f} at the start, "
          f"{step_misses[-1]:,.2f} after {len(step_misses) - 1} epochs")

## Dial: Neurons and Layers

- **Few rows:** 20 of the curve’s 300 points (every 15th) to fit; the
  other 280 held out. The miss is on the held-out points (against the
  points, not the true curve: never below the noise).
- **Too few** neurons: too stiff, misses the bends. **Too many:** they
  hold far more numbers than there are points, and draw through every
  point.
- **A layer** bends the bends of the layer before; the count of numbers
  grows with width × width.
- More neurons: fewer stuck starts, but slower to train.

**Try it.** The three networks of this slide on the dials’ 20 points:
`few_network`, `fitting_miss` and `held_out_miss` are in the second
Setup cell. The 2 layers of 100 memorize the 20 points, so their misses
can differ a little, in the third decimal, from the slide’s and from one
computer to another. Then try `(5,)` or `(100,)`.

In [ ]:
# Three sizes, each trained 600 epochs on the 20 points from start 0, no penalty.
for hidden_layers in [(2,), (20,), (100, 100)]:
    size_network = few_network(hidden_layers)
    print(f"{str(hidden_layers):>10}: held-out miss {held_out_miss(size_network):.3f}, "
          f"fitting miss {fitting_miss(size_network):.3f}")

## Dial: How Long to Train

- **The gap opens:** the fitting miss falls toward zero; the held-out
  miss falls, then climbs.
- **Early stopping:** hold out 10% of the fitting rows; stop when their
  miss stops improving. For returns, hold out the latest months.
- In scikit-learn: `early_stopping=True`; it waits a few epochs without
  improvement, then keeps the best epoch’s weights.

**Try it.** The 2 layers of 100 of the last cell, trained one epoch at a
time (`partial_fit`) and scored after some epochs on the 20 points it
fits and on the 280 held out. Past the best epoch, its numbers can
differ a little from the slide’s.

In [ ]:
# The 2 layers of 100, no penalty, start 0: one epoch at a time, 600 epochs.
long_network = MLPRegressor(hidden_layer_sizes=(100, 100), alpha=0.0, learning_rate_init=0.01,
                            random_state=0)
for epoch in range(1, 601):
    long_network.partial_fit(few_X, few_y)                      # one more epoch
    if epoch in [10, 39, 100, 300, 600]:
        print(f"epoch {epoch:>3}: fitting miss {fitting_miss(long_network):.3f}, "
              f"held-out miss {held_out_miss(long_network):.3f}")

## Dial: The Penalty

- **Ridge’s penalty, on every weight:** the squared gaps plus λ × the
  sum of squared weights. It pulls the weights toward zero; the curve
  calms down.
- **No penalty:** through every point. **A middle one:** smooth. **A big
  one:** almost a straight line.
- In scikit-learn: `alpha`. Search it over powers of ten, on validation
  data.

**Try it.** The same 2 layers of 100 at the three penalties of this
slide. λ = 0 is the memorizing network, so its miss can differ a little
from the slide’s. Then try 0.01 or 1.

In [ ]:
# The 2 layers of 100 on the 20 points, 600 epochs, at three penalties λ (alpha in scikit-learn).
for penalty_value in [0.0, 0.1, 3.0]:
    penalty_network = few_network((100, 100), penalty=penalty_value)
    print(f"λ = {penalty_value:g}: held-out miss {held_out_miss(penalty_network):.3f}")

## Dial: Random Starts

- **With few neurons the start decides:** the same network, from another
  random start, misses by twice as much.
- **Averaging the starts** helps where they disagree, barely where they
  agree: it removes the start’s noise, not the data’s.

**Try it.** 10 neurons on the 20 points from five random starts, each
scored on the 280 held-out points, then the five forecasts averaged.
Then try 20 neurons.

In [ ]:
# 10 neurons, five random starts (random_state), 600 epochs each, no penalty.
forecasts_by_start = []
for start in range(5):
    start_network = few_network((10,), start=start)
    forecasts_by_start.append(start_network.predict(held_X))
    print(f"start {start}: held-out miss {held_out_miss(start_network):.3f}")
average_forecast = np.mean(forecasts_by_start, axis=0)        # the five forecasts, averaged
print(f"the five averaged: held-out miss {np.mean((average_forecast - held_y) ** 2):.3f}")

## Why Tuning a Network Is Harder

| Model         | Dials that matter |
|---------------|-------------------|
| Random forest | 2                 |
| Boosted trees | 3                 |
| Network       | 6                 |

- **More dials, multiplied:** neurons, layers, step, epochs, λ, batch
  size, and the starts. Each fit is slower than a tree’s. A forest does
  well out of the box.
- **They interact:** a bigger network wants a smaller step, fewer epochs
  or a larger λ. Change one, and the best value of another moves.
- **The start moves the score** as much as a dial does.
- **In practice:** fix most dials; tune two or three on validation data.

## Check: A Jumpy Miss

- A network’s miss on its fitting rows, printed after every epoch, jumps
  up and down, and grows.
- **Fingers up:** which dial do you change first?
  1.  Lower the learning rate
  2.  Add neurons
  3.  Train longer
  4.  Add a layer

# Networks on Real Data

## Predict: The Network Joins the Race

In [ ]:
# The race of the trees deck ("The Race on Real Data"): five models, each scored on homes it
# did not see (5-fold cross-validation). This cell prepares it and takes no time; the next cell
# runs it. The Your Turn needs this cell and the one that defines homes_network, not the race.
# A blank in a feature column (no alley, no pool) means "none", so categories get "none".
word_columns = homes.select_dtypes(exclude="number").columns.tolist()
number_columns = [name for name in homes.columns if name not in word_columns]

dummies = make_pipeline(SimpleImputer(strategy="constant", fill_value="none"),
                        OneHotEncoder(handle_unknown="ignore"))
for_lines = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), number_columns),
    ("words", dummies, word_columns)])
for_trees = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), number_columns), ("words", dummies, word_columns)])
folds = KFold(5, shuffle=True, random_state=0)


def homes_r2(forecasts):
    """R², %, of out-of-fold forecasts of log price: the forecast for each home comes from a
    model that never saw that home."""
    return 100 * (1 - np.sum((log_price - forecasts) ** 2)
                  / np.sum((log_price - log_price.mean()) ** 2))

In [ ]:
# The race itself: about a minute on Colab.
racers = {
    "linear regression": make_pipeline(for_lines, LinearRegression()),
    "ridge": make_pipeline(for_lines, RidgeCV(alphas=np.logspace(-1, 3, 30))),
    "one tree, depth 4": make_pipeline(for_trees, DecisionTreeRegressor(
        max_depth=4, min_samples_leaf=10, random_state=0)),
    "random forest": make_pipeline(for_trees, RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, max_features=0.33, n_jobs=-1, random_state=0)),
    "boosted trees": make_pipeline(for_trees, GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=3, subsample=0.8, random_state=0)),
}

homes_forecasts = {}                         # each model's out-of-fold forecast of log price
for name, racer in racers.items():
    homes_forecasts[name] = cross_val_predict(racer, homes, log_price, cv=folds)
homes_score = {name: homes_r2(forecast) for name, forecast in homes_forecasts.items()}
for name, score in homes_score.items():
    print(f"{name:18s} R² {score:5.1f}%")

- **The homes:** 1,460 sales, 79 features, log price. The cell above
  prints each model’s $R^2$ on homes it did not see.
- **A network joins:** scikit-learn’s default size and penalty (one
  hidden layer of 100 neurons), early stopping on, nothing tuned; three
  random starts, their forecasts averaged.
- **Predict first:** where does the average land?
  1.  Ahead of boosting
  2.  Between boosting and the forest
  3.  Between the forest and ridge
  4.  Below ridge

## The Network on the Homes

In [ ]:
# The network that joins the race: defined here, run in the next cell and in the Your Turn.
def homes_network(start, layers=(100,), penalty=0.0001):
    """A network for log price: scikit-learn's default unless told otherwise (one hidden layer
    of 100 neurons; penalty λ = 0.0001, which scikit-learn calls alpha). Inputs prepared as for
    ridge (for_lines: numbers standardized, categories as 0/1 columns). early_stopping: hold
    out 10% of the fitting rows and stop when their miss stops improving.
    TransformedTargetRegressor standardizes log price inside each fit and turns the forecasts
    back into log price: a network starts near zero, and log price is about 12."""
    network = MLPRegressor(hidden_layer_sizes=layers, alpha=penalty, early_stopping=True,
                           random_state=start)
    return TransformedTargetRegressor(make_pipeline(for_lines, network),
                                      transformer=StandardScaler())

In [ ]:
start_forecasts = {}                       # the same 5 folds as the race; a few seconds
for start in [0, 1, 2]:
    start_forecasts[start] = cross_val_predict(homes_network(start), homes, log_price,
                                               cv=folds)
start_scores = {start: homes_r2(forecast) for start, forecast in start_forecasts.items()}
network_score = homes_r2(np.mean(list(start_forecasts.values()), axis=0))   # average of 3

rows = {**homes_score, "network, average of 3": network_score}
rows["network, each start"] = np.mean(list(start_scores.values()))      # for the order only
order = sorted(rows, key=rows.get)                                       # lowest at the bottom
colors = {"linear regression": "gray", "ridge": "gray", "one tree, depth 4": "tab:blue",
          "random forest": "tab:orange", "boosted trees": "tab:orange"}
plt.figure(figsize=(10, 3.4))
for position, name in enumerate(order):
    if name == "network, each start":         # three dots, one per start
        low, high = min(start_scores.values()), max(start_scores.values())
        plt.plot([low, high], [position, position], color="tab:green", linewidth=2)
        plt.scatter(list(start_scores.values()), [position] * 3, color="tab:green", s=60, zorder=3)
        plt.text(high + 0.5, position, f"{low:.1f} to {high:.1f}%", va="center")
    else:
        plt.barh(position, rows[name], color=colors.get(name, "tab:green"))
        plt.text(rows[name] + 0.5, position, f"{rows[name]:.1f}%", va="center")
plt.yticks(range(len(order)), order)
plt.xlim(60, 100)
plt.xlabel("R² on homes the model did not see (%)")
plt.show()

- **Default size and penalty, early stopping on, nothing tuned.** The
  start alone moves the score.
- **Their average** beats every start and the forest; boosting stays
  ahead.
- **79 features, 1,460 homes:** a table that suits trees. Networks shine
  with far more data and raw inputs (pictures, text).

## Your Turn: Change the Network

- Fill the blanks: the network’s layers and its penalty λ. Run one
  start; compare with the default.

Before you paste it: run the cell that prepares the race (the first cell
under “Predict: The Network Joins the Race”) and the cell that defines
`homes_network`. The race itself need not run.

``` python
layers = ____         # neurons per layer: (16,) or (256,)
penalty = ____        # λ: 0.0001 (the default), 1 or 10
network = homes_network(0, layers, penalty)
forecast = cross_val_predict(network, homes, log_price,
                             cv=folds)
print(f"R² {homes_r2(forecast):.1f}%")
```

## Check: Three Starts

- Three random starts of the same network score differently on the very
  homes they are judged on.
- **Fingers up:** what is best practice?
  1.  Average the three forecasts
  2.  Keep the best start
  3.  Keep the first start
  4.  Add layers until the starts agree

# A Finance Application

## Which Stocks Will Do Better?

- **Gu, Kelly & Xiu (2020):** can a network tell which stocks will do
  better next month?
- **One row:** one stock in one month. **The target:** its return the
  next month, over the T-bill.
- **Almost 30,000 US stocks,** 1957–2016: over 6,200 in a typical month.
- **94 features (firm characteristics):** size, value, momentum,
  volatility, trading volume, … Each month, each is ranked from −1 (the
  lowest stock) to +1 (the highest); a blank gets the middle, 0.
- **920 features in all:** each characteristic alone and times each of 8
  market-wide numbers (the T-bill rate, …), plus 74 industry dummies.
- **The score:** $R^2$ on years no model saw, against a forecast of
  **zero**: stricter than a stock’s own past average, which is so noisy
  that it is easy to beat.

## The Split in Time

- **Fit** on the past, **choose the dials** on the 12 years just before,
  **forecast** one year no model has seen.
- **Then move a year** and refit: one more fitting year; the 12
  validation years roll forward. 30 refits forecast 1987–2016.
- **No shuffling:** the rows keep their order in time. Early stopping
  watches the validation years.

## Five Networks, One Recipe

- **One to five hidden layers,** each half as wide as the last: 32, 16,
  8, 4, 2 neurons, all with the hockey-stick bend.
- **About 30,000 numbers** each, almost all of them between the 920
  features and the first 32 neurons.
- **The recipe:** the squared miss plus lasso’s penalty λ; Adam, a
  walker that adjusts each weight’s step as it goes; stop when the
  validation years stop improving; average 10 random starts.
- **Shapes fixed in advance;** only λ and the step were tuned. A search
  over shapes: “unrealistic and unnecessary”.

## Who Forecasts Best

- **OLS on all 920 features: −3.46%,** worse than forecasting zero.
  Penalties help, and so do PCR and PLS (regressions on a few
  combinations of the features); curves in each feature alone add
  little.
- **Trees and networks lead,** with interactions. Best: three layers,
  0.40%; four and five layers do no better.
- **Is the gap real?** Over OLS on three features, yes (two to five
  layers); over the trees, no: within the noise.

## What the Networks Use

- **Importance:** set one feature to the middle (0), keep the fitted
  model, and see how much $R^2$ falls.
- **All their models agree** on a short list: price trends, then
  liquidity, volatility, valuation. Linear models lean on one or two;
  trees and networks spread out.
- **It depends on size:** last month’s winners fall back and losers
  bounce, strongly for small stocks; for large stocks, mostly the
  winners fall back.

## Tiny R², Real Money

- **Each month:** sort the stocks into tenths by the three-layer
  forecast; buy the top tenth, sell the bottom (value-weighted). Sharpe
  ratio: **1.20** a year.
- **Four layers did best, 1.35;** OLS on size, value and momentum, 0.61.
  Timing the S&P 500 with three layers: 0.77, against 0.51 for buy and
  hold.
- **Why it pays:** one stock’s month is mostly noise; across a tenth of
  the market the noise averages out. **Before trading costs,** with
  turnover over 100% a month.

## Check: Why Not Deeper?

- Gu, Kelly & Xiu fitted networks with one to five hidden layers.
- Their $R^2$ peaked at three layers; four and five did no better.
- **Fingers up:** the most likely reason?
  1.  Too little data and too weak a signal for more layers
  2.  One hidden layer can already draw any curve, so more cannot help
  3.  Deeper networks cannot be trained by walking downhill
  4.  Stock returns have no interactions

# Networks With Memory

## Order Matters

- **One market, four months** (made-up returns, oldest first: +1%, −1%,
  +2%, −20%, and the reverse) in two orders: the same average and the
  same standard deviation.
- **How risky is the market now?** After a crash last month: high, since
  big months come in clusters. A running volatility, which weighs recent
  months more, tells the two orders apart.
- **Words too:** “the dog bit the man” and “the man bit the dog”: the
  same words, opposite news.

## Lag Columns or a Running Summary

- **One series, the market’s monthly return:** to forecast next month
  (say, how risky it is), a network reads the past months in one of two
  ways.

|  | **Lag columns** (a plain network) | **A running summary** (a recurrent network) |
|:------------------|:--------------------------|:--------------------------|
| Months it reads | Exactly 12, one column each | Any number, one month at a time |
| A crash 1 or 7 months back | Two different weights, each learned on its own | One rule, the same every month |
| A longer history | More columns, more weights | The same weights |

- **A recurrent network** reads the months in order, oldest first, and
  carries a running summary, its **state**. “Recurrent”: the same step
  recurs every month.
- **Text the same way:** a sentence of any length, word by word.

## A Running Summary: Volatility

In [ ]:
# The stock market's monthly return over T-bills, in percent, 1926 to 2025 (Goyal-Welch).
stock_return = 100 * monthly["equity"].dropna()


def running_volatility(returns, keep):
    """A running summary of how big returns have been. Each month: keep a share of the old
    state, and fill the rest with this month's squared return. Volatility = its square root."""
    state = returns.iloc[0] ** 2                  # start: the first month's squared return
    states = []
    for one_return in returns:
        state = keep * state + (1 - keep) * one_return ** 2   # the same weight, every month
        states.append(state)
    return pd.Series(np.sqrt(states), index=returns.index)


quick = running_volatility(stock_return, keep=0.5)
smooth = running_volatility(stock_return, keep=0.97)

figure, axes = plt.subplots(1, 2, figsize=(11, 2.6))
for axis, (first, last) in zip(axes, [("1926", "1945"), ("2000", "2025")]):
    size = stock_return[first:last].abs()          # how big each month's return was
    axis.vlines(size.index, 0, size, color="lightgray", linewidth=1.5)
    axis.plot(quick[first:last], color="tab:orange", linewidth=1.8, label="keep 0.5")
    axis.plot(smooth[first:last], color="tab:blue", linewidth=2.8, label="keep 0.97")
    axis.set_title(f"{first} to {last}")
    decades = [pd.Timestamp(f"{year}-01-01") for year in range(1930, 2030, 10)]
    axis.set_xticks([day for day in decades if first <= str(day.year) <= last])
    axis.set_xticklabels([str(day.year) for day in decades if first <= str(day.year) <= last])
axes[0].set_ylabel("% a month")
axes[0].legend(loc="upper right", frameon=False, fontsize=15)
plt.subplots_adjust(wspace=0.15)
plt.show()

- **Each month:** summary = keep × last month’s summary + (1 − keep) ×
  this month’s squared return; volatility = √summary. Gray: each month’s
  return, its size.
- **The simplest recurrent network:** one neuron, no bend, fed the
  squared return; one dial, keep. 0.97 is smooth but slow; 0.5 is quick
  but jumpy.
- **In a recurrent network, keep is a weight,** learned by the walk
  downhill; RiskMetrics sets it by hand.

## A Recurrent Network, Step by Step

- **The state:** a running summary the network writes itself. Each month
  it reads its last state and this month’s input: the size of the
  market’s move (made-up months: +1%, −2%, +1%, −20%, −6%, +2%).
- **One step:** 0.5 × last month’s state + 0.05 × this month’s move,
  then **squashed**. The squash: an S-curve from −1 to 1, which keeps
  the state in that range.
- **The same weights every month** (0.5, 0.05, and a constant of 0
  here), learned by the walk downhill. A crash fades fast: each month
  keeps only half of the last state.

**Try it.** The plain recurrent network of this slide, month by month:
one line in a loop, the same weights every month. Then try 0.9 instead
of 0.5 on the last state: the crash fades more slowly.

In [ ]:
# One neuron with made-up weights: 0.5 on the last state, 0.05 on the size of this month's move,
# then the squash, tanh. The six made-up months, oldest first.
state = 0.0                                         # before the first month
for month, one_return in enumerate([1, -2, 1, -20, -6, 2], start=1):
    state = np.tanh(0.5 * state + 0.05 * abs(one_return))     # the same weights every month
    print(f"month {month}: return {one_return:+d}%, state {state:.2f}")

## Why Blame Fades

- **Training passes the blame back** month by month (backpropagation,
  “How a Network Learns”). Each month back multiplies it by the weight
  on the last state, 0.5 here, and by the squash’s slope, at most 1.
- **0.5 multiplied 20 times is about one millionth:** a crash 20 months
  back teaches the weights almost nothing.
- **A multiplier near 1 lets blame travel** much further. The LSTM
  builds such a path.

## The LSTM: Two Lines

- **LSTM** (long short-term memory; Hochreiter & Schmidhuber 1997)
  splits the plain network’s one state in two lines.
- **The memory line** (the **cell state**): a running summary kept
  inside the cell, changed only by multiplying and adding.
- **The shown state** (the **hidden state**): what the cell puts out
  each month, to the forecast and to next month’s step.

## A Gate: A Valve Between 0 and 1

- **A gate is one neuron:** a weighted sum of what it reads, this
  month’s input and last month’s shown state, plus a constant.
- **Then the S-bend,** the logit’s curve from the trees deck: any sum
  in, a number between 0 and 1 out (the squash runs from −1 to 1).
- **Its number multiplies something:** 0 closes the gate (nothing
  passes), 1 opens it (all passes), 0.3 lets 30% through.

## The Forget Gate: Keep How Much

- **The forget gate** reads this month’s input and last month’s shown
  state, and **multiplies the memory** by its number, 0 to 1.
- **1 keeps all of the memory; 0 wipes it.** A gate of 0.9 keeps 90% of
  it; a gate of 0.1, 10%.
- **The running volatility’s keep, now set by the data:** near 1 in calm
  months, near 0 when a crash arrives.

## The Input Gate: Write How Much

- **The candidate** is this month’s new reading, from the same two
  inputs, squashed between −1 and 1. **The input gate**, 0 to 1, says
  how much of it to write.
- **Their product is added** to the memory. New memory = forget gate ×
  old memory + input gate × candidate.

## The Output Gate: Show How Much

- **The output gate**, 0 to 1, reads the same two inputs and says how
  much of the memory to show.
- **Shown state = output gate × squashed memory.**
- **The memory stays inside;** only the shown state goes on, to the
  forecast and to next month’s gates.

## An LSTM by Hand: Six Months

- **Made-up months and weights.** One LSTM cell through the six months
  of the recurrent network’s example: the gates, candidate, memory and
  shown state are worked month by month.
- **The crash:** the forget gate falls to about 0, wiping the calm past;
  the input gate opens to about 1; the memory jumps.

**Try it.** The LSTM cell of the table, month by month: the three gates
and the candidate, then the memory and the shown state. Then try another
crash, such as 40 for the fourth month’s move.

In [ ]:
# One LSTM cell through the same six months, with the table's made-up weights. Each part reads
# only the size of this month's move (in a real LSTM, also last month's shown state).
def s_bend(total):
    """The gates' S-bend: any sum in, a number between 0 and 1 out."""
    return 1 / (1 + np.exp(-total))


memory = 0.0                                        # before the first month
for month, move_size in enumerate([1, 2, 1, 20, 6, 2], start=1):
    forget_gate = s_bend(3 - 0.5 * move_size)       # how much of the memory to keep
    input_gate = s_bend(-2 + 0.5 * move_size)       # how much of the candidate to write
    candidate = np.tanh(0.1 * move_size)            # this month's new reading
    output_gate = s_bend(1.5 + 0.2 * move_size)     # how much of the memory to show
    memory = forget_gate * memory + input_gate * candidate      # keep some, write some
    shown = output_gate * np.tanh(memory)                       # show some
    print(f"month {month}: forget {forget_gate:.2f}, input {input_gate:.2f}, "
          f"memory {memory:.2f}, shown {shown:.2f}")

## What the Gates Did

- **The keep now moves:** about 0.9 in calm months, about 0 in the
  crash, one half the month after, then back near 0.9.
- **The memory holds the crash** in the months after it.
- **The running volatility’s summary is an LSTM memory line with frozen
  gates:** forget = keep, input = 1 − keep, and the squared return,
  unsquashed, as the candidate. The LSTM lets the data move the gates.

## Networks With Memory in Finance

- **A panel, stock by stock:** one LSTM, with the same weights for every
  stock, reads each S&P 500 stock’s last 240 daily returns and ranks the
  stocks for the next day (Fischer & Krauss 2018).
- **Many series at once:** an LSTM condenses 178 macroeconomic series
  into a few states that feed a model of stock prices (Chen, Pelger &
  Zhu 2024).
- **One series, as here:** volatility, where gates can shorten the
  memory after a shock, as the running summary cannot.

## Check: What a Gate Does

- At one step, an LSTM’s forget gate outputs 0.2.
- **Fingers up:** the memory line then:
  1.  Keeps 20% of its memory
  2.  Forgets 20% of its memory
  3.  Adds 20% new input
  4.  Shrinks the learning rate by 20%

# What Networks Can Do That Trees Cannot

## Predict: Past the Edge

- Boosted trees and a network, each fitted to 300 noisy points of a
  straight trend, with x between −1 and 1.
- Each is then asked for a forecast at x = 2, where there are no data.
- **Predict first:** what do the boosted trees forecast at x = 2?
  1.  The trend continued: about 2
  2.  The same as at the edge of the data
  3.  Zero: no data, no forecast
  4.  The average of all 300 points

## Past the Edge of the Data

- **Boosted trees go flat:** past the edge of the data, each tree
  repeats its edge leaf.
- **The network keeps going** in a straight line, near its slope at the
  edge: close to the trend, far below the curve once the curve turns.
- **Neither knows the world past the data:** on the trend the network is
  closer, on the curve the trees. In finance: a month wilder than any in
  the fitting years.

**Try it.** The straight trend of the left panel: boosted trees and a
network, each fitted to its 300 points, then asked about x at the edge
of the data and past it. Then try 3 or −2.

In [ ]:
# 300 noisy points of a straight trend, y = x, with x between -1 and 1.
trend_draw = np.random.default_rng(7)
x_trend = np.sort(trend_draw.uniform(-1, 1, 300))
y_trend = x_trend + trend_draw.normal(0, 0.3, 300)
X_trend = x_trend.reshape(-1, 1)
trend_trees = GradientBoostingRegressor(random_state=0).fit(X_trend, y_trend)
trend_network = MLPRegressor(hidden_layer_sizes=(100,), learning_rate_init=0.01, max_iter=5000,
                             random_state=0).fit(X_trend, y_trend)
for x_new in [1.0, 1.5, 2.0]:                       # the data end just below 1
    trees_say = trend_trees.predict([[x_new]])[0]
    network_says = trend_network.predict([[x_new]])[0]
    print(f"x = {x_new}: boosted trees {trees_say:.2f}, network {network_says:.2f}, "
          f"truth {x_new:.2f}")

## Raw Inputs, Learned Features

- **Trees, and plain networks, see 64 unrelated columns.** Moved one
  pixel, a handwritten 0 is a new row: most of its 64 numbers change. On
  these tiny grids that hardly matters; on real photos it does.
- **A convolutional network** slides small detectors over the picture:
  it finds a stroke wherever it sits, then builds strokes into shapes.
- **Words, sound, price paths:** recurrent networks read sequences and
  learn what to carry forward. Nobody hand-builds these features; the
  network learns them.

**Try it.** The plain network of “A Picture Is a Grid of Numbers”,
fitted again, on the quarter of the digits it never saw. In the file,
every digit is written in about the same place, so the network reads
almost all of them. Move each one pixel to the right, as an object moves
in a photo, and it reads about half.

In [ ]:
# The digits network, fitted again (fit_digit_network is in the second Setup cell), and the
# digits it never saw, each as a grid of 8 x 8 squares.
digits_new, labels_new, digit_network = fit_digit_network()
new_grids = digits_new.reshape(-1, 8, 8)
moved_grids = np.zeros_like(new_grids)
moved_grids[:, :, 1:] = new_grids[:, :, :-1]        # every column one pixel to the right
moved_rows = moved_grids.reshape(-1, 64)            # back to 64 numbers a digit
print(f"New digits read right: {digit_network.score(digits_new, labels_new):.0%}")
print(f"The same digits, moved one pixel: {digit_network.score(moved_rows, labels_new):.0%}")

## One Middle, Many Uses

- **The middle layers are features the network learned.** Each output
  gets its own last layer to weigh them, and what one output teaches the
  middle helps the others: lanes, cars and road signs, all at once.
- **Pre-train, then tune:** learn the middle on a huge data set, keep
  it, then keep walking downhill on a small one. A million labeled
  photos, then medical scans; web text, then following instructions
  (ChatGPT).

## Where Trees Still Win

- **Tables of a few dozen features:** boosted trees usually match or
  beat networks. On the homes, boosting stayed ahead of the averaged
  network. Stock returns: a tie within the noise.
- **Small data:** a network has many numbers to fit (about 30,000 in
  each of Gu, Kelly & Xiu’s), so it needs many rows.
- **Easier tuning:** two or three dials that matter, and defaults that
  work. A network has many dials; they interact; the start moves the
  score.
- **Less preparation:** trees ask questions, so the scale of a feature
  does not matter. A network needs its inputs standardized.

# Close

## The Rest, Briefly

- **Convolutional networks,** the detectors of the digits, also read
  price charts as images (Jiang, Kelly & Xiu 2023).
- **Autoencoders** squeeze many inputs through a narrow layer: many
  returns into a few factors (Gu, Kelly & Xiu 2021). More with factor
  models.
- **Embeddings** turn words into numbers, so that similar words sit
  close together. More with text.
- **Large language models:** transformers trained to predict the next
  word, on vast amounts of text. More with text.

## Networks in One Slide

- **A network adds up hockey sticks:** with enough of them, any
  continuous curve over a bounded range.
- **It learns by walking downhill:** backpropagation passes the blame
  back to every weight, one batch at a time.
- **The dials:** the step, neurons and layers, how long to train, the
  penalty λ, batch size, random starts. More dials than a forest, harder
  to tune.
- **On the homes,** the untuned network landed between the forest and
  boosting. **Gu, Kelly & Xiu:** shallow networks forecast returns a
  little better than trees, within the noise.
- **Recurrent networks** carry a state; the LSTM’s gates decide what to
  keep, write and show.
- **When to reach for one:** big data and raw inputs (pictures, text,
  sound). A table of a few dozen features: boosted trees first.

## Readings

**Required**

- Gu, Kelly & Xiu (2020), “Empirical Asset Pricing via Machine
  Learning”, *Review of Financial Studies* 33(5), 2223–2273: the
  introduction and Tables 1, 6 and 7.
- James, Witten, Hastie, Tibshirani & Taylor (2023), *ISLP*, sections
  10.1–10.3 and 10.5–10.7 (networks, convolution, memory, when to use
  them, fitting).

**Optional**

- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*,
  sections 11.3–11.5.
- Prince (2023), *Understanding Deep Learning*, chapters 3, 6, 7 and 10;
  free at udlbook.github.io.